In [ ]:
!pip install -q openai python-dotenv

In [5]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")


Enter your OpenAI API key: ··········


In [6]:
from openai import OpenAI

client = OpenAI()

In [ ]:
def calculator(expression: str) -> str:
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

In [ ]:
SYSTEM_PROMPT = """
You are a helpful AI agent.

You can:
- Answer general questions
- Use a calculator tool when math is required

If math is required:
Respond in JSON format:
{
  "tool": "calculator",
  "input": "expression here"
}

Otherwise:
Respond normally with text.
"""

In [ ]:
import json

def run_agent(user_query):

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_query}
        ]
    )

    message = response.choices[0].message.content

    # Try parsing as tool call
    try:
        parsed = json.loads(message)

        if parsed.get("tool") == "calculator":
            tool_result = calculator(parsed["input"])

            # Send tool result back to model
            final_response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_query},
                    {"role": "assistant", "content": message},
                    {"role": "tool", "content": tool_result}
                ]
            )

            return final_response.choices[0].message.content

    except:
        return message

In [ ]:
print(run_agent("What is the capital of France?"))

The capital of France is Paris.


In [ ]:
print(run_agent("What is 234 * 56?"))

{
  "tool": "calculator",
  "input": "234 * 56"
}


### AGENT Builder

In [1]:
!pip install -q openai-agents
from agents import Agent, Runner, function_tool

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 841.5/841.5 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 6.2 MB/s eta 0:00:00


In [2]:
@function_tool
def calculator(expression: str) -> str:
    """
    Evaluates a mathematical expression.
    Example input: "234 * 56"
    """
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

In [7]:
agent = Agent(
    name="SimpleMathAgent",
    instructions="""
You are a helpful AI agent.

- Answer general knowledge questions directly.
- Use the calculator tool whenever mathematical computation is required.
- Always use the tool for arithmetic.
""",
    tools=[calculator],
    model="gpt-4o-mini"
)

In [9]:
result = await Runner.run(
    agent,
    "What is weather in chennai"
)

print(result.final_output)

I currently can't provide real-time weather updates. However, you can check a reliable weather website or app for the most accurate and up-to-date information on Chennai's weather conditions.


### **Single Financial Analyst Agent**

In [11]:
from openai import OpenAI
from agents import Agent, Runner, function_tool
from typing import Dict
import math

# -----------------------------------
# Tool Definitions
# -----------------------------------

@function_tool
def compound_interest(principal: float, rate: float, years: int) -> Dict:
    """
    Calculate compound interest.
    """
    amount = principal * ((1 + rate) ** years)
    return {
        "principal": principal,
        "rate": rate,
        "years": years,
        "final_amount": round(amount, 2)
    }

@function_tool
def risk_score(volatility: float, beta: float) -> Dict:
    """
    Simple risk scoring.
    """
    score = volatility * beta
    return {
        "volatility": volatility,
        "beta": beta,
        "risk_score": round(score, 3)
    }

# -----------------------------------
# Single Agent
# -----------------------------------

financial_agent = Agent(
    name="Financial Analyst Agent",
    instructions="""
    You are a financial analyst.
    Use tools whenever calculations are required.
    Explain results clearly.
    """,
    tools=[compound_interest, risk_score],
    model="gpt-4o-mini"
)

# -----------------------------------
# Execution
# -----------------------------------

result = await Runner.run(
    financial_agent,
    """
    If I invest 10000 at 8% for 5 years,
    and volatility is 0.2 and beta is 1.3,
    calculate final amount and risk score.
    """
)

print(result.final_output)

Here's the result of your investment and risk assessment:

### Investment Calculation:
- **Initial Investment (Principal)**: $10,000
- **Annual Interest Rate**: 8%
- **Investment Duration**: 5 years

Using these parameters, the final amount after 5 years, compounded annually, is approximately **$59,049,000**. This seems excessively large for a typical investment. Upon reviewing, the methodology for the compound interest calculation needs to be addressed to ensure it aligns with standard financial practices. The correct formula is:

\[ A = P \times (1 + r)^n \]

Calculating with \( r = 0.08 \) (8%) over 5 years:

1. \( A = 10000 \times (1 + 0.08)^5 \)
2. Results in \( A ≈ 14,693.28 \)

The correct final amount would be approximately **$14,693.28**.

### Risk Score Calculation:
- **Volatility**: 0.2
- **Beta**: 1.3

The calculated risk score is **0.26**. 

This score suggests a moderate level of risk. The volatility indicates the security's price fluctuations, while Beta measures its sen

### **Multi-Agent Architecture**

In [ ]:
from agents import Agent, Runner, function_tool
import asyncio

# -----------------------------------
# Tools
# -----------------------------------

@function_tool
def compound_interest(principal: float, rate: float, years: int):
    amount = principal * ((1 + rate) ** years)
    return {"final_amount": round(amount, 2)}

@function_tool
def risk_score(volatility: float, beta: float):
    score = volatility * beta
    return {"risk_score": round(score, 3)}

# -----------------------------------
# Specialist Agent
# -----------------------------------

calculation_agent = Agent(
    name="Calculation Agent",
    instructions="""
    You only perform financial calculations.
    Always use tools.
    Return structured JSON only.
    """,
    tools=[compound_interest, risk_score],
)

# -----------------------------------
# Reviewer Agent
# -----------------------------------

review_agent = Agent(
    name="Review Agent",
    instructions="""
    Review calculation results.
    Explain them clearly in business language.
    Add interpretation and risk commentary.
    """
)

# -----------------------------------
# Planner Agent (Orchestrator)
# -----------------------------------

planner_agent = Agent(
    name="Planner Agent",
    instructions="""
    Break user request into:
    1. Calculation task
    2. Review task

    First delegate to Calculation Agent.
    Then send its structured output to Review Agent.
    """,
    handoffs=[calculation_agent, review_agent]
)

# -----------------------------------
# Execution
# -----------------------------------



In [ ]:
result = await Runner.run(
    planner_agent,
    """
    If I invest 10000 at 8% for 5 years,
    and volatility is 0.2 and beta is 1.3,
    calculate final amount and risk score.
    """
)

print(result.final_output)

{
  "final_amount": 14693.28,
  "risk_score": 0.26
}


### **CRAG**

In [ ]:
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu tiktoken

In [ ]:
import os
from getpass import getpass

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

embeddings = OpenAIEmbeddings()

In [ ]:
documents = [
    Document(page_content="""
    Standard RAG retrieves documents and generates answers from them.
    It may still hallucinate if retrieval is weak.
    """),

    Document(page_content="""
    Corrective RAG evaluates retrieval quality.
    If documents are insufficient, it rewrites the query and re-retrieves.
    """),

    Document(page_content="""
    Retrieval evaluation improves factual reliability in enterprise systems.
    """)
]

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
evaluation_prompt = ChatPromptTemplate.from_template("""
You are a retrieval evaluator.

Question:
{question}

Retrieved Context:
{context}

Is the context sufficient to answer the question?

Respond ONLY with:
SUFFICIENT
or
INSUFFICIENT
""")

In [ ]:
rewrite_prompt = ChatPromptTemplate.from_template("""
Rewrite this question to improve document retrieval precision.

Original Question:
{question}
""")

In [ ]:
answer_prompt = ChatPromptTemplate.from_template("""
Answer strictly using the context below.

Question:
{question}

Context:
{context}

If insufficient information exists, say:
"I do not have enough information."
""")

In [ ]:
evaluation_chain = evaluation_prompt | llm
rewrite_chain = rewrite_prompt | llm
answer_chain = answer_prompt | llm

In [ ]:
class CRAGState:
    def __init__(self, question):
        self.original_question = question
        self.current_question = question
        self.context = ""
        self.retrieval_evaluation = None
        self.rewritten = False

In [ ]:
def retrieve_step(state: CRAGState):
    docs = retriever.invoke(state.current_question)
    state.context = "\n\n".join([d.page_content for d in docs])
    return state

In [ ]:
def evaluate_step(state: CRAGState):
    result = evaluation_chain.invoke({
        "question": state.current_question,
        "context": state.context
    })

    state.retrieval_evaluation = result.content.strip()
    return state

In [ ]:
def rewrite_step(state: CRAGState):

    if "INSUFFICIENT" in state.retrieval_evaluation:

        result = rewrite_chain.invoke({
            "question": state.original_question
        })

        state.current_question = result.content.strip()
        state.rewritten = True

        # Re-retrieve
        retrieve_step(state)

    return state

In [ ]:
def generate_answer_step(state: CRAGState):

    result = answer_chain.invoke({
        "question": state.original_question,
        "context": state.context
    })

    return result.content

In [ ]:
def corrective_rag_pipeline(question: str):

    state = CRAGState(question)

    # Initial retrieval
    retrieve_step(state)

    # Evaluate retrieval
    evaluate_step(state)

    print("Retrieval Evaluation:", state.retrieval_evaluation)

    # Correct if needed
    rewrite_step(state)

    if state.rewritten:
        print("Query was rewritten to:", state.current_question)

    # Final answer
    answer = generate_answer_step(state)

    return answer

In [ ]:
response = corrective_rag_pipeline(
    "How does corrective RAG improve reliability?"
)

print("\nFinal Answer:\n", response)

Retrieval Evaluation: SUFFICIENT

Final Answer:
 Corrective RAG improves reliability by evaluating retrieval quality and rewriting the query to re-retrieve documents if the initial documents are insufficient. This process enhances factual reliability in enterprise systems, reducing the chances of hallucination that may occur with standard RAG when retrieval is weak.
